<a href="https://colab.research.google.com/github/ichaxpro/244107020023-Machine-Learning/blob/master/JS09/JS09_Lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# -------------------------------
# Load dataset
# -------------------------------
# Pastikan file CSV berada di direktori yang sama atau sesuaikan path-nya
df = pd.read_csv('songs_with_attributes_and_lyrics.csv')
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
X = df[features].values

# Feature standardization
scaler = StandardScaler()
# Ubah ke float32 karena FAISS dan HNSW membutuhkan format 32-bit array
X_scaled = scaler.fit_transform(X).astype('float32')

k = 10  # number of nearest neighbors
num_samples, dim = X_scaled.shape

# Menentukan satu lagu acak sebagai data query (misalnya lagu pada index 0)
query_idx = 0
query_vector = X_scaled[query_idx].reshape(1, -1) # klearn/faiss
query_vector_1d = query_vector[0]                 # annoy

print(f"Mencari {k} lagu terdekat untuk lagu pada indeks: {query_idx}...")
print("-" * 50)

results = []

# ---------------------------------------------------------
# 1. Exact NN (Scikit-Learn) - Brute Force
# ---------------------------------------------------------
nn_exact = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn_exact.fit(X_scaled)

start_time = time.time()
distances_exact, indices_exact = nn_exact.kneighbors(query_vector)
exact_time = time.time() - start_time

results.append({"Algorithm": "Exact NN (Sklearn)", "Time (s)": exact_time})
print("1. Exact NN")
print(f"   Waktu Query : {exact_time:.6f} detik")
print(f"   Indeks Hasil: {indices_exact[0]}\n")


# ---------------------------------------------------------
# 2. ANNOY (Approximate Nearest Neighbors Oh Yeah)
# ---------------------------------------------------------
annoy_index = AnnoyIndex(dim, 'euclidean')
for i in range(num_samples):
    annoy_index.add_item(i, X_scaled[i])
annoy_index.build(20) # Menggunakan 20 trees untuk akurasi

start_time = time.time()
indices_annoy = annoy_index.get_nns_by_vector(query_vector_1d, k)
annoy_time = time.time() - start_time

results.append({"Algorithm": "ANNOY", "Time (s)": annoy_time})
print("2. ANNOY")
print(f"   Waktu Query : {annoy_time:.6f} detik")
print(f"   Indeks Hasil: {indices_annoy}\n")


# ---------------------------------------------------------
# 3. FAISS (IndexFlatL2 - Exact Search as baseline in FAISS)
# ---------------------------------------------------------
faiss_index = faiss.IndexFlatL2(dim)
faiss_index.add(X_scaled)

start_time = time.time()
distances_faiss, indices_faiss = faiss_index.search(query_vector, k)
faiss_time = time.time() - start_time

results.append({"Algorithm": "FAISS (FlatL2)", "Time (s)": faiss_time})
print("3. FAISS")
print(f"   Waktu Query : {faiss_time:.6f} detik")
print(f"   Indeks Hasil: {indices_faiss[0]}\n")


# ---------------------------------------------------------
# 4. HNSW (Hierarchical Navigable Small World)
# ---------------------------------------------------------
# Inisialisasi index (space='l2' merepresentasikan jarak Euclidean squared)
hnsw_index = hnswlib.Index(space='l2', dim=dim)
hnsw_index.init_index(max_elements=num_samples, ef_construction=200, M=16)
hnsw_index.add_items(X_scaled)
hnsw_index.set_ef(50) # Parameter query time

start_time = time.time()
indices_hnsw, distances_hnsw = hnsw_index.knn_query(query_vector, k=k)
hnsw_time = time.time() - start_time

results.append({"Algorithm": "HNSW", "Time (s)": hnsw_time})
print("4. HNSW")
print(f"   Waktu Query : {hnsw_time:.6f} detik")
print(f"   Indeks Hasil: {indices_hnsw[0]}\n")


# ---------------------------------------------------------
# Rangkuman Hasil
# ---------------------------------------------------------
print("=== RANGKUMAN KOMPARASI WAKTU PENCARIAN ===")
df_results = pd.DataFrame(results)
print(df_results.to_markdown(index=False))

C:\Users\icha\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Mencari 10 lagu terdekat untuk lagu pada indeks: 0...
--------------------------------------------------
1. Exact NN
   Waktu Query : 0.487446 detik
   Indeks Hasil: [     0 394553 764272 837727 749223  61511 625227 833164 676733 523698]

2. ANNOY
   Waktu Query : 0.000591 detik
   Indeks Hasil: [0, 394553, 764272, 837727, 61511, 625227, 833164, 676733, 523698, 763625]

3. FAISS
   Waktu Query : 0.013297 detik
   Indeks Hasil: [     0 394553 764272 837727 749223  61511 625227 833164 676733 523698]

4. HNSW
   Waktu Query : 0.000651 detik
   Indeks Hasil: [     0 394553 764272 837727 749223  61511 625227 833164 676733 523698]

=== RANGKUMAN KOMPARASI WAKTU PENCARIAN ===
| Algorithm          |    Time (s) |
|:-------------------|------------:|
| Exact NN (Sklearn) | 0.487446    |
| ANNOY              | 0.000590563 |
| FAISS (FlatL2)     | 0.0132971   |
| HNSW               | 0.00065136  |
